# Delta-V budget — how it works

This is the explanation and reference for the `quicksat` delta-V budget: what the manoeuvre file holds, which closed forms turn each row into a delta-V, how its single margin differs from the mass budget's two layers, and where the tool stops.

It is deliberately not a tutorial. In [Diátaxis](https://diataxis.fr/) terms, `sample/` holds the tutorials and how-to guides; `docs/` holds the explanation and the reference. It still runs, against the same sample data, because an explanation that cannot be executed drifts from the code it describes.

The premise throughout: **Mission Analysis owns the orbital mechanics.** This module holds their numbers and does the small closed-form arithmetic that turns each into a delta-V. Anything needing real analysis arrives as a `given` manoeuvre with the answer already worked out.

In [1]:
import os
import tempfile
from pathlib import Path

import pandas as pd

# Make the in-development quicksat package importable without installing it: walk up
# from the current directory to the repo root (the folder that holds the quicksat
# package) and switch to it. Works whether the notebook runs from docs/, the repo
# root, or the docs build.
here = Path.cwd()
repo_root = next(
    (p for p in (here, *here.parents) if (p / "quicksat" / "__init__.py").exists()),
    None,
)
if repo_root is None:
    raise RuntimeError("could not locate the quicksat repo root")
os.chdir(repo_root)

from quicksat import Q_, R_EARTH
from quicksat.delta_v.budget import (
    DeltaVBudget,
    deorbit_deltav,
    hohmann_deltav,
    plane_change_deltav,
)
from quicksat.utils.orbit import Orbit

pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

DATA = Path("sample") / "data"
orbit = Orbit.from_yaml_file(DATA / "orbit.yaml")
budget = DeltaVBudget.from_csv(
    DATA / "manoeuvres.csv", DATA / "delta_v_config.yaml", DATA / "orbit.yaml"
)

## The data model

This is a table, like the mass budget and unlike the data budget. There is a list of things — manoeuvres — and the budget is their sum, so the input is a CSV and `phase` and `manoeuvre_type` are two ordinary columns to group over.

What differs from the mass budget is how the margin is applied. There is **one margin on the total**, not a per-item contingency plus a system margin. The reason is where the uncertainty actually sits: a Hohmann transfer's delta-V is a closed form and is known to more figures than anyone needs, while *how many collision avoidance manoeuvres a year* is a guess that could be out by a factor of two. Contingency on each manoeuvre would decorate the precise part and leave the imprecise part bare.

In [2]:
budget.manoeuvre_table

,manoeuvre_id,manoeuvre_name,phase,manoeuvre_type,value,count,recurring,comments
0,injection_correction,Launcher dispersion correction,Commissioning,altitude_change,12 kilometer,1.000,False,Semi-major axis dispersion at separation
1,inclination_trim,Injection inclination trim,Commissioning,inclination_change,0.05 degree,1.000,False,Plane error at separation
2,phasing,Orbit phasing to the reference slot,Commissioning,given,8.0 meter / second,1.000,False,From Mission Analysis; needs a real phasing an...
3,drag_makeup,Drag make-up,Operations,altitude_change,1.2 kilometer,1.000,True,Per year at 500 km solar mean
4,collision_avoidance,Collision avoidance,Operations,collision_avoidance,200 meter,4.000,True,Per year; one-way hop as the drag make-up abso...
5,inclination_maint,Inclination maintenance,Operations,inclination_change,0.012 degree,1.000,True,"Per year, holds the sun-synchronous plane"
6,deorbit,End-of-life deorbit,Disposal,deorbit,250 kilometer,1.000,False,Single burn to a 500 x 250 km disposal orbit; ...


## The manoeuvre file

| column | meaning |
|---|---|
| `manoeuvre_id` | short identifier, no whitespace (`collision_avoidance`) |
| `manoeuvre_name` | full name, free text |
| `phase` | mission phase — the reporting axis (`Commissioning`, `Operations`, `Disposal`) |
| `manoeuvre_type` | which closed form applies |
| `value` | the input, with its unit; what it *means* depends on the type |
| `count` | how many times |
| `recurring` | when set, `count` is per year and is multiplied by the mission duration |
| `comments` | free text |

### One value column, five meanings

A manoeuvre's input is a length for an altitude change, an angle for a plane change, and a velocity for something Mission Analysis computed elsewhere. Rather than four mostly-empty columns, there is one `value` column and the type decides what dimension it must carry:

| type | `value` is | delta-V |
|---|---|---|
| `altitude_change` | altitude delta | Hohmann between the two circular altitudes |
| `collision_avoidance` | altitude offset | the same Hohmann, doubled when the config says the hop returns |
| `inclination_change` | angle | `2·v·sin(Δi/2)` |
| `deorbit` | target perigee altitude | one impulse: `v·(1 − √(2r_p/(r+r_p)))` |
| `given` | the delta-V itself | taken as-is |

That check is the whole point of collapsing four columns into one. It is a model validator rather than a field one, because the required dimension depends on another field in the same row:

In [3]:
HEADER = "manoeuvre_id,manoeuvre_name,phase,manoeuvre_type,value,count,recurring,comments"


def load_row(label, row):
    """Build a one-row budget from a CSV line, and report what the loader makes of it."""
    path = Path(tempfile.mkdtemp()) / "manoeuvres.csv"
    path.write_text(f"{HEADER}\n{row}\n")
    try:
        DeltaVBudget.from_csv(path, DATA / "delta_v_config.yaml", DATA / "orbit.yaml")
        print(f"{label:34s} accepted")
    except ValueError as exc:
        reason = str(exc).split("Value error, ")[-1].split(" [type=")[0]
        print(f"{label:34s} rejected - {reason}")


load_row("a plain altitude change", "x,X,Ops,altitude_change,1 km,1,false,")
load_row("velocity where a length belongs", "x,X,Ops,altitude_change,8 m/s,1,false,")
load_row("length where a velocity belongs", "x,X,Ops,given,200 m,1,false,")
load_row("length where an angle belongs", "x,X,Ops,inclination_change,5 km,1,false,")
load_row("an angle, correctly", "x,X,Ops,inclination_change,0.05 deg,1,false,")
load_row("a whitespace id", "a b,X,Ops,given,8 m/s,1,false,")
load_row("a negative count", "x,X,Ops,given,8 m/s,-1,false,")

a plain altitude change            accepted
velocity where a length belongs    rejected - A 'altitude_change' manoeuvre needs a length in its value column, got '8.0 meter / second'
length where a velocity belongs    rejected - A 'given' manoeuvre needs a velocity in its value column, got '200 meter'
length where an angle belongs      rejected - A 'inclination_change' manoeuvre needs an angle in its value column, got '5 kilometer'
an angle, correctly                accepted
a whitespace id                    rejected - /tmp/tmp_d7h3odm/manoeuvres.csv, row 2:
1 validation error for Manoeuvre
manoeuvre_id
  String should match pattern '^\S+$'
a negative count                   rejected - /tmp/tmpfoo0ywxg/manoeuvres.csv, row 2:
1 validation error for Manoeuvre
count
  Input should be greater than or equal to 0


## The closed forms

Three of them, all assuming circular orbits and impulsive burns.

**Hohmann transfer** between two circular radii — two impulses, one to leave and one to circularise:

$$\Delta v = v_1\left|\sqrt{\tfrac{2r_2}{r_1+r_2}} - 1\right| + v_2\left|1 - \sqrt{\tfrac{2r_1}{r_1+r_2}}\right|$$

**Plane change** at circular velocity, which is why it is expensive: $\Delta v = 2v\sin(\Delta i / 2)$.

**Deorbit**, a single impulse lowering perigee — to a disposal altitude where drag is left to finish the job, or to the surface for a direct re-entry: $\Delta v = v\left(1 - \sqrt{\tfrac{2r_p}{r+r_p}}\right)$.

In [4]:
print(f"circular velocity at {orbit.altitude:~.0f}:  {orbit.velocity:~.4f}")
print()
for delta in ("200 m", "1 km", "12 km"):
    hop = hohmann_deltav(orbit.radius, orbit.radius + Q_(delta))
    print(f"  raise by {delta:6s}  {hop:~8.4f}")

disposal = deorbit_deltav(orbit.radius, R_EARTH + Q_(250, "km"))
reentry = deorbit_deltav(orbit.radius, R_EARTH)
print(f"\ndeorbit to a 250 km perigee  {disposal:~.2f}")
print(f"deorbit to the surface       {reentry:~.2f}")
print("  the same single impulse; the target perigee is the biggest lever there is")

# a plane change is priced at orbital velocity, so a fraction of a degree is dear
trim = plane_change_deltav(orbit.velocity, Q_(0.05, "deg"))
print(f"\n0.05 deg of plane change {trim:~.3f}")
print(f"  which is what raising the orbit by 12 km costs "
      f"({hohmann_deltav(orbit.radius, orbit.radius + Q_(12, 'km')):~.3f})")

circular velocity at 500 km:  7.6126 km / s

  raise by 200 m     0.1107 m / s
  raise by 1 km      0.5533 m / s
  raise by 12 km     6.6320 m / s

deorbit to a 250 km perigee  70.78 m / s
deorbit to the surface       144.95 m / s
  the same single impulse; the target perigee is the biggest lever there is

0.05 deg of plane change 6.643 m / s
  which is what raising the orbit by 12 km costs (6.632 m / s)


### Collision avoidance has no physics of its own

A collision avoidance manoeuvre is a hop: raise the orbit by a couple of hundred metres, let the conjunction pass, come back down. That is two Hohmann transfers, so the type reuses the altitude-change primitive and multiplies by two.

The config's `return_burn` turns the doubling off. That is not a modelling shortcut but a real operational case: when the satellite was already due to be raised against its drag debt, the avoidance hop does double duty and there is nothing to undo.

In [5]:
one_way = hohmann_deltav(orbit.radius, orbit.radius + Q_(200, "m"))
frame = budget.resolve().set_index("manoeuvre_id")

print(f"one-way 200 m hop        {one_way:~.4f}")
charged = frame.loc["collision_avoidance", "deltav_each"]
print(f"the budget's CAM line    {charged:.4f} m/s")
print(
    f"  return_burn is {budget.config.collision_avoidance.return_burn}, "
    f"so it is charged {charged / one_way.magnitude:.0f}x the one-way hop"
)

one-way 200 m hop        0.1107 m / s
the budget's CAM line    0.1107 m/s
  return_burn is False, so it is charged 1x the one-way hop


## Counts, and the mission duration

`count` is how many times a manoeuvre happens. When `recurring` is set it is a **rate** — that many per year — and the mission duration turns it into a total. Everything else happens once.

That split is why the mission duration lives in the config rather than being folded into the counts: extending a 7 year mission to 10 is one number, not a pass over every row.

In [6]:
resolved = budget.resolve()
resolved[["manoeuvre_id", "count", "recurring", "occurrences", "deltav_each", "deltav_total"]]

,manoeuvre_id,count,recurring,occurrences,deltav_each,deltav_total
0,injection_correction,1.000,False,1.000,6.632,6.632
1,inclination_trim,1.000,False,1.000,6.643,6.643
2,phasing,1.000,False,1.000,8.000,8.000
3,drag_makeup,1.000,True,7.000,0.664,4.648
4,collision_avoidance,4.000,True,28.000,0.111,3.099
5,inclination_maint,1.000,True,7.000,1.594,11.161
6,deorbit,1.000,False,1.000,70.783,70.783


## The margin

One allowance, applied once, to the total. It is the last thing that happens, so both grouped views carry it proportionally and still reconcile to the same number.

In [7]:
for margin in (False, True):
    total = budget.total_deltav(margin)
    phases = budget.by_phase(margin)["deltav"].sum()
    types = budget.by_type(margin)["deltav"].sum()
    label = f"margin={margin}"
    print(f"{label:14s} total {total:~8.2f}   by phase {phases:8.2f}   by type {types:8.2f}")

print(f"\nthe margin is {budget.config.margin:g}% of the budget, or "
      f"{(budget.total_deltav() - budget.total_deltav(False)):~.2f}")

margin=False   total   110.97 m / s   by phase   110.97   by type   110.97


margin=True    total   116.51 m / s   by phase   116.51   by type   116.51

the margin is 5% of the budget, or 5.55 m / s


## The rocket equation, and the sizing loop

A delta-V budget's purpose is eventually a propellant mass. Given the dry mass and the exhaust velocity $v_e = I_{sp}g_0$:

$$m_{prop} = m_{dry}\left(e^{\Delta v / v_e} - 1\right)$$

The argument is the **dry** mass — the final mass of the burn — so the propellant is what has to be added on top. Passing the wet mass instead would answer a different question and give a smaller number.

`propellant_mass` takes that mass as an argument rather than reaching for a `MassBudget`. The two modules stay decoupled, and the loop is closed in a notebook where it is visible. Nothing is written back: the equipment CSV remains the source of truth for what is actually loaded, and the comparison is left to a human.

In [8]:
from quicksat.mass.budget import MassBudget

mass_data = MassBudget.from_csv(DATA / "equipment.csv", DATA / "budget_config.yaml")
dry = mass_data.in_orbit_mass(propellant=0)

print(f"dry mass             {dry:~.2f}")
print(f"delta-V              {budget.total_deltav():~.2f}")
print(f"propellant required  {budget.propellant_mass(dry):~.2f}")
print(f"propellant loaded    {mass_data.propellant_mass():~.2f}")
print()
print("the two disagree, which is the loop doing its job: the equipment list was")
print("written before the delta-V budget existed, and has not been revised to match")

dry mass             448.62 kg
delta-V              116.51 m / s


propellant required  24.89 kg
propellant loaded    22.00 kg

the two disagree, which is the loop doing its job: the equipment list was
written before the delta-V budget existed, and has not been revised to match


## The document view

`tabulated_deltav()` lays the budget out as a document: manoeuvres grouped into mission phases with a subtotal each, then the total, the margin, and the total including it. Phases come out in the order the file lists them rather than sorted, which for a manoeuvre file is usually chronological and reads as the mission does.

Same Styler conventions as the other two reports: blanks rather than NaN, bold summaries, `row_type` present in `.data` but hidden in the render, and `comments` hidden unless asked for.

In [9]:
budget.tabulated_deltav()

Item,Name,Type,Input,ΔV each [m/s],Times,ΔV total [m/s]
injection_correction,Launcher dispersion correction,altitude_change,12 km,6.632,1.0,6.63
inclination_trim,Injection inclination trim,inclination_change,0.05 deg,6.643,1.0,6.64
phasing,Orbit phasing to the reference slot,given,8 m / s,8.000,1.0,8.00
,Commissioning subtotal,,,,,21.28
drag_makeup,Drag make-up,altitude_change,1.2 km,0.664,7.0,4.65
collision_avoidance,Collision avoidance,collision_avoidance,200 m,0.111,28.0,3.10
inclination_maint,Inclination maintenance,inclination_change,0.012 deg,1.594,7.0,11.16
,Operations subtotal,,,,,18.91
deorbit,End-of-life deorbit,deorbit,250 km,70.783,1.0,70.78
,Disposal subtotal,,,,,70.78


In [10]:
data = budget.tabulated_deltav().data
print(data["row_type"].value_counts().to_dict())

summary = data[data["row_type"].isin(["subtotal", "margin", "total"])]
print()
print(summary[["name", "deltav"]].to_string(index=False))

{'manoeuvre': 7, 'phase_subtotal': 3, 'subtotal': 1, 'margin': 1, 'total': 1}

                name  deltav
Total, before margin 110.966
         Margin (5%)   5.548
            Total ΔV 116.515


## Limitations

What the delta-V budget deliberately does not do:

- **Impulsive burns only.** No finite-burn losses, no gravity losses, no thrust-to-weight check. A low-thrust electric transfer is not this model at all — it would need a different formulation, not a correction factor.
- **Circular orbits, two-impulse transfers.** No eccentricity anywhere, so no bi-elliptic transfers, no combined manoeuvres, and no plane change flown at apogee where it would be cheaper.
- **No perturbations.** The drag make-up figure is an input, not something computed from an atmosphere model and a ballistic coefficient. So is the inclination maintenance.
- **One propulsion system, one Isp.** No cold-gas-plus-monopropellant split, no blowdown curve, no thruster cant or cosine losses.
- **The rocket equation ignores staging and residuals.** No unusable propellant, no pressurant, no margin for a failed burn that has to be repeated.
- **No epoch, no sequencing.** Manoeuvres have counts, not dates. Nothing models a conjunction rate that rises with the debris environment, or a deorbit that gets cheaper because drag has already lowered the orbit.
- **The margin is a flat percentage.** Not a statistical combination, and applied to the total rather than to the uncertain parts, which is conservative and is meant to be.